In [ ]:
import pandas as pd
from glob import glob


import librosa, librosa.display
from pydub import AudioSegment
import IPython.display as ipd
import whisper
import plotly.express as px
from sklearn.decomposition import PCA
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
import json
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from speechbrain.inference.speaker import EncoderClassifier
from sklearn.manifold import TSNE
import os

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
print(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / 'data'
WAV_DIR = DATA_DIR / 'wav'
M4A_DIR = DATA_DIR / 'm4a'
IMAGE_DIR = PROJECT_ROOT / 'images'
TRANSCRIPTION_DIR = DATA_DIR / 'transcription'
DEEPFILTERNET = DATA_DIR / 'deepfilternet'

In [ ]:
audio_files_m4a = glob(f'{M4A_DIR}/*.m4a')
print(len(audio_files_m4a))

In [ ]:
y, sr = librosa.load(audio_files_m4a[5])
pd.Series(y).plot(title='Audio Signal')

In [ ]:
print(f'Shape of audio signal: {y.shape}')

In [ ]:
data_frame_m4a = []

In [ ]:
for f in audio_files_m4a:
    y, sr = librosa.load(f)
    data_frame_m4a.append({
        'file': f,
        'duration_s': librosa.get_duration(y=y, sr=sr),
        'sample_rate': sr,
        'num_samples': len(y),
        'file_format': f.split('.')[-1],
    })
df = pd.DataFrame(data_frame_m4a)

In [ ]:
df

In [ ]:
df.describe()

### Convert to .wav format

In [ ]:
for f in audio_files_m4a:
    base_file_name = f.split('/')[-1].split('.')[0]
    audio = AudioSegment.from_file(f, format='m4a')
    audio.export(f'data/wav/{base_file_name}.wav', format='wav')

### Signal Analysis


In [ ]:
audio_files_wav = glob(f'{WAV_DIR}/*.wav')
print(len(audio_files_wav))

In [ ]:
data_frame_wav = []
for f in audio_files_wav:
    y, sr = librosa.load(f)
    data_frame_wav.append({
        'file': f,
        'duration_s': librosa.get_duration(y=y, sr=sr),
        'sample_rate': sr,
        'num_samples': len(y),
        'file_format': f.split('.')[-1],
    })
df_wav = pd.DataFrame(data_frame_wav)

In [ ]:
peaks = []
for f in audio_files_wav:
    y, sr = librosa.load(f)
    peak = np.max(np.abs(y))
    peaks.append(peak)
df_wav['peak_amplitude'] = peaks

In [ ]:
df_wav['peak_amplitude'].hist(bins = 42)

In [ ]:
df_filtered = df_wav[df_wav['peak_amplitude'] > 0.9]
df_filtered

Note: Some of the audio files have very high peak amplitudes. After listening to the files, it seems that these are mostly background noise or silent recordings. Remember for the preprocessing to filter out the background noise.

In [ ]:
audio_file = glob(f'{WAV_DIR}/*.wav')
ipd.Audio(audio_file[0])

In [ ]:
y, sr = librosa.load(audio_file[0])
librosa.display.waveshow(y, sr=sr)

### Spectrogram analysis

The goal of the mel spectrogram analysis is to visualize the noises and quality degradation. This prevents the model from learning the background noise and instead focuses on cloning the patient’s natural and clear voice identity.



In [ ]:
audio_file = glob(f'{WAV_DIR}/*.wav')
ipd.Audio(audio_file[0])

In [ ]:
n_fft = 2048
hop_length = 512

for files in audio_files_wav:
    signal, sr = librosa.load(files)

    base_file_name = os.path.splitext(os.path.basename(files))[0]
    mel_signal = librosa.feature.melspectrogram(y=signal, sr=sr, hop_length=hop_length,
                                                n_fft=n_fft)
    spectrogram = np.abs(mel_signal)

    power_to_db = librosa.power_to_db(spectrogram, ref=np.max)
    plt.figure(figsize=(10,4))
    librosa.display.specshow(power_to_db, sr=sr, x_axis='time', y_axis='mel', cmap='magma',
                             hop_length=hop_length)
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'Mel-Spectrogram (dB) of {base_file_name}')
    plt.xlabel('Time', fontdict=dict(size=15))
    plt.ylabel('Frequency', fontdict=dict(size=15))
    plt.tight_layout()
    plt.savefig(f'{IMAGE_DIR}/{base_file_name}.png')
    plt.close()

### Speaker Embedding

The goal is to generate a speaker embedding and project it into a 2D space for visualization. This will help to understand the speaker's voice characteristics. The expectation is that all embeddings will be similar to each other.

In [ ]:
print("Lade Speaker-Modell...")
classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")

# After applying deepfilternet


In [ ]:
print(WAV_DIR)
audio_files = glob(f'{DEEPFILTERNET}/*.wav')
audio_files.sort()

audio_embeddings = []
file_labels = []

for f in audio_files:
    signal, sr = librosa.load(f)

    if sr != 16000:
        signal = librosa.resample(signal, orig_sr=sr, target_sr=16000)

    singnal_tensor = torch.tensor(signal, dtype=torch.float32, device=classifier.device)

    with torch.no_grad():
        embeddings = classifier.encode_batch(singnal_tensor)

    single_embedding = embeddings.squeeze().cpu().numpy()
    audio_embeddings.append(single_embedding)
    file_labels.append(os.path.basename(f))

all_embeddings = np.array(audio_embeddings)

In [ ]:
embeddings_array = np.array(all_embeddings)

pca = PCA(n_components=2)
reduced_embeddings = pca.fit_transform(embeddings_array)

df_pca_embeddings = pd.DataFrame(reduced_embeddings, columns=['PC1', 'PC2'])
df_pca_embeddings['file'] = [os.path.basename(f) for f in audio_files]

fig = px.scatter(df_pca_embeddings, x='PC1', y='PC2',
                 title='PCA of audio Embeddings')
fig.show()

### Phoneme Analysis

The goal of the phoneme analysis is to understand the phonetic characteristics of the audio files. I want to ensure that all relevant sounds and phonemes are present in the audio files. This will help if the model struggels withspecific words.


In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model_id = "primeline/whisper-large-v3-german"
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)
processor = AutoProcessor.from_pretrained(model_id)
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    max_new_tokens=128,
    chunk_length_s=30,
    batch_size=16,
    return_timestamps=True,
    torch_dtype=torch_dtype,
    device=device,
)

In [ ]:
all_transcriptions = []

for audio_file_path in audio_files:
    full_path = os.path.join(WAV_DIR, audio_file_path)
    result = pipe(full_path)
    transcription = result["text"]

    file_id = os.path.basename(audio_file_path)

    all_transcriptions.append({
        "id": file_id,
        "path": full_path,
        "transcription": transcription
    })

output_json_path = f'{TRANSCRIPTION_DIR}/transcriptions.json'
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(all_transcriptions, f, ensure_ascii=False, indent=4)